# 🧪 Backend End-to-End Pipeline & Unit Testing
This notebook allows you to test each backend service and function sequentially:
1. **Configuration & Settings** (`.env` validation)
2. **MongoDB Connection & CRUD** (`app.database`)
3. **Text Extraction Service** (`app.services.text_extraction`)
4. **Embedding Service with AICredits Gateway** (`app.services.embeddings`)
5. **End-to-End Ingestion Pipeline** (Equipment -> Doc Metadata -> Text Splitting -> Embeddings -> MongoDB Storage & Vector Verification)
6. **Cleanup** (Optional)

## 1. Environment & Configuration Check

In [1]:
import os
import sys
from pathlib import Path

# Add backend root to path
backend_dir = Path(os.getcwd())
if str(backend_dir) not in sys.path:
    sys.path.insert(0, str(backend_dir))

from app.config import settings

print("=== Configuration Loaded ===")
print(f"📁 Working Dir: {os.getcwd()}")
print(f"🏢 Tenant ID: {settings.TENANT_ID}")
print(f"👤 User ID: {settings.USER_ID}")
print(f"🗄️  DB Name: {settings.DB_NAME}")
print(f"🌐 Gateway Base URL: {settings.BASE_URL}")
print(f"🧠 Embedding Model: {settings.EMBEDDING_MODEL}")
print(f"✂️  Chunk Size / Overlap: {settings.CHUNK_SIZE} / {settings.CHUNK_OVERLAP}")
print(f"📦 Vector Index: {settings.VECTOR_INDEX_NAME}")
print(f"📦 Document Chunks Collection: {settings.DOCUMENT_CHUNKS_COLLECTION}")
print(f"🔑 API Key Loaded: {'✅ Yes' if settings.API_KEY else '❌ No'}")
print(f"🍃 Mongo URI Loaded: {'✅ Yes' if settings.MONGO_URI else '❌ No'}")

=== Configuration Loaded ===
📁 Working Dir: \\wsl.localhost\Ubuntu\home\dell\voice-agent\backend
🏢 Tenant ID: mvp_tenant
👤 User ID: mvp_user
🗄️  DB Name: live_db
🌐 Gateway Base URL: https://aicredits.in/v1
🧠 Embedding Model: baai/bge-m3
✂️  Chunk Size / Overlap: 1000 / 250
📦 Vector Index: vector_index
📦 Document Chunks Collection: document_chunks
🔑 API Key Loaded: ✅ Yes
🍃 Mongo URI Loaded: ✅ Yes


## 2. Test MongoDB Connection

In [2]:
from app.database import connect_to_mongo, get_database, close_mongo_connection

# Connect to MongoDB
await connect_to_mongo()
db = get_database()

# Ping server
ping_res = await db.command('ping')
print("✅ MongoDB Ping Result:", ping_res)

# List collections in DB
collections = await db.list_collection_names()
print(f"📂 Existing Collections in '{settings.DB_NAME}':", collections)

2026-08-15 18:52:57.369 | INFO     | app.database:connect_to_mongo:12 - Connecting to MongoDB
2026-08-15 18:53:07.825 | INFO     | app.database:connect_to_mongo:17 - Connected to MongoDB


✅ MongoDB Ping Result: {'ok': 1}
📂 Existing Collections in 'live_db': ['equipment', 'document_chunks', 'documents_metadata']


## 3. Test Text Extraction Service

In [3]:
import tempfile
from app.services.text_extraction import TextExtractionService

extractor = TextExtractionService()

# 1. Verify format detection
print("Format support checks:")
print(" - .txt:", extractor.is_supported("text/plain", "sample.txt"))
print(" - .pdf:", extractor.is_supported("application/pdf", "sample.pdf"))
print(" - .docx:", extractor.is_supported("application/vnd.openxmlformats-officedocument.wordprocessingml.document", "sample.docx"))
print(" - .exe (should be False):", extractor.is_supported("application/octet-stream", "sample.exe"))

# 2. Test extraction from plain text file
with tempfile.NamedTemporaryFile(delete=False, suffix=".txt", mode="w", encoding="utf-8") as tmp:
    tmp.write("Turbine Maintenance Standard Operating Procedure\n1. Check vibration sensor.\n2. Inspect lube oil pressure.")
    sample_path = tmp.name

extracted = extractor.extract_text(sample_path, "text/plain")
print("\nExtracted text from sample file:")
print("---")
print(extracted)
print("---")
os.remove(sample_path)

2026-08-15 18:54:16.040 | DEBUG    | app.services.text_extraction:is_supported:113 - Checking file support
2026-08-15 18:54:16.041 | DEBUG    | app.services.text_extraction:is_supported:119 - Supported via content_type
2026-08-15 18:54:16.041 | DEBUG    | app.services.text_extraction:is_supported:113 - Checking file support
2026-08-15 18:54:16.041 | DEBUG    | app.services.text_extraction:is_supported:119 - Supported via content_type
2026-08-15 18:54:16.042 | DEBUG    | app.services.text_extraction:is_supported:113 - Checking file support
2026-08-15 18:54:16.042 | DEBUG    | app.services.text_extraction:is_supported:119 - Supported via content_type
2026-08-15 18:54:16.042 | DEBUG    | app.services.text_extraction:is_supported:113 - Checking file support
2026-08-15 18:54:16.043 | WARNING  | app.services.text_extraction:is_supported:125 - Unsupported file type
2026-08-15 18:54:16.045 | INFO     | app.services.text_extraction:extract_text:17 - Extracting text from file
2026-08-15 18:54:16

Format support checks:
 - .txt: True
 - .pdf: True
 - .docx: True
 - .exe (should be False): False

Extracted text from sample file:
---
Turbine Maintenance Standard Operating Procedure
1. Check vibration sensor.
2. Inspect lube oil pressure.
---


## 4. Test Embedding Service (AICredits Gateway & Splitting)

In [4]:
from app.services.embeddings import EmbeddingService

embedding_service = EmbeddingService()

# 1. Test splitting text into chunks
sample_manual = """
1. Introduction to Gas Turbine Operation
The gas turbine is an internal combustion engine that uses air as the working fluid.
It extracts chemical energy from fuel and converts it to mechanical energy.

2. Pre-Start Checklist
Ensure auxiliary lube oil pump is running and discharge pressure is at least 2.5 bar.
Verify cooling water supply temperature is between 25C and 35C.
Check hydraulic trip circuit reset status on the HMI control panel.

3. Operational Monitoring
Continuously monitor exhaust gas temperatures (EGT) for spread deviations.
Inspect radial bearing vibrations with peak-to-peak velocity limit below 4.5 mm/s.
""" * 4

chunks = embedding_service.split_text(sample_manual)
print(f"✂️  Original text: {len(sample_manual)} characters")
print(f"✂️  Split into {len(chunks)} chunks")
print(f"Chunk #1 preview: {chunks[0][:120]}...")

2026-08-15 18:54:22.337 | INFO     | app.services.embeddings:__init__:11 - Initializing EmbeddingService
2026-08-15 18:54:22.346 | INFO     | app.services.embeddings:__init__:29 - EmbeddingService ready
2026-08-15 18:54:22.348 | DEBUG    | app.services.embeddings:split_text:37 - Splitting text into chunks
2026-08-15 18:54:22.349 | INFO     | app.services.embeddings:split_text:42 - Text split completed


✂️  Original text: 2520 characters
✂️  Split into 4 chunks
Chunk #1 preview: 1. Introduction to Gas Turbine Operation
The gas turbine is an internal combustion engine that uses air as the working f...


In [5]:
# 2. Test single chunk embedding (AICredits Gateway)
test_query = "Turbine lube oil pressure inspection test."
single_vector = embedding_service.embed_text(test_query)

print("✅ Single embedding succeeded!")
print(f"🔢 Vector Dimension: {len(single_vector)}")
print(f"📊 Vector preview (first 5 floats): {single_vector[:5]}")

2026-08-15 18:54:29.007 | DEBUG    | app.services.embeddings:embed_text:46 - Embedding single chunk
2026-08-15 18:54:31.292 | DEBUG    | app.services.embeddings:embed_text:55 - Chunk embedded successfully


✅ Single embedding succeeded!
🔢 Vector Dimension: 1024
📊 Vector preview (first 5 floats): [-0.0304414052516222, 0.0078085619024932384, -0.03488500416278839, -0.01843816228210926, 0.003088392084464431]


In [6]:
# 3. Test batch embedding (AICredits Gateway)
batch_inputs = [
    "Check auxiliary lube oil pump pressure.",
    "Verify cooling water supply temperature.",
    "Inspect radial bearing vibration levels."
]

batch_vectors = embedding_service.embed_texts(batch_inputs)
print("✅ Batch embedding succeeded!")
print(f"🔢 Embedded {len(batch_vectors)} chunks")
print(f"🔢 Dimension of each chunk: {len(batch_vectors[0])}")

2026-08-15 18:54:34.070 | INFO     | app.services.embeddings:embed_texts:59 - Embedding batch of texts
2026-08-15 18:54:34.967 | INFO     | app.services.embeddings:embed_texts:79 - Batch embedding complete


✅ Batch embedding succeeded!
🔢 Embedded 3 chunks
🔢 Dimension of each chunk: 1024


## 5. End-to-End Simulation: Equipment -> Document -> Chunks in MongoDB

In [7]:
import uuid
from datetime import datetime, timezone
from bson import ObjectId

# 1. Create a test Equipment record
equipment_name = f"Gas Turbine Unit #{uuid.uuid4().hex[:6].upper()}"
equipment_data = {
    "name": equipment_name,
    "tenant_id": settings.TENANT_ID,
    "created_at": datetime.now(timezone.utc),
    "updated_at": datetime.now(timezone.utc)
}
eq_result = await db.equipment.insert_one(equipment_data)
equipment_id = eq_result.inserted_id
print(f"✅ Equipment created: ID = {equipment_id}, Name = '{equipment_name}'")

✅ Equipment created: ID = 6a80689674408963e3a0f887, Name = 'Gas Turbine Unit #7DA453'


In [9]:
# 2. Prepare document text, split into chunks, and compute embeddings
doc_filename = "turbine_maintenance_sop.txt"
doc_content = """
# Gas Turbine Operating Guide
Section 1: Daily inspections on oil levels and filter differential pressures.
Section 2: Weekly tests on flame scanners and spark plugs.
Section 3: Monthly calibration of temperature and vibration transducers.
"""

# Split
doc_chunks = embedding_service.split_text(doc_content)
print(f"Generated {len(doc_chunks)} chunks for document '{doc_filename}'")

# Insert Document Metadata
now = datetime.now(timezone.utc)
doc_record = {
    "equipment_id": equipment_id,
    "tenant_id": settings.TENANT_ID,
    "file_name": doc_filename,
    "content_type": "text/plain",
    "size": len(doc_content.encode('utf-8')),
    "storage_key": f"{settings.TENANT_ID}/equipment/{equipment_id}/{uuid.uuid4().hex}-{doc_filename}",
    "uploaded_by": settings.USER_ID,
    "description": "Test SOP manual for Gas Turbine Unit",
    "document_type": "knowledge",
    "embedding_status": "processing",
    "created_at": now,
    "updated_at": now,
}
doc_meta_res = await db.documents_metadata.insert_one(doc_record)
document_id = doc_meta_res.inserted_id
print(f"✅ Document metadata inserted: ID = {document_id}")

# Generate embeddings and prepare chunk records
chunk_records = []
for idx, chunk_text in enumerate(doc_chunks):
    emb = embedding_service.embed_text(chunk_text)
    chunk_records.append({
        "document_id": document_id,
        "equipment_id": equipment_id,
        "tenant_id": settings.TENANT_ID,
        "file_name": doc_filename,
        "chunk_id": str(uuid.uuid4()),
        "chunk_index": idx,
        "text": chunk_text,
        "embedding": emb,
        "is_disabled": False,
    })

# Insert chunks
await db[settings.DOCUMENT_CHUNKS_COLLECTION].insert_many(chunk_records)

# Update document metadata to completed
await db.documents_metadata.update_one(
    {"_id": document_id},
    {"$set": {"embedding_status": "completed", "updated_at": datetime.now(timezone.utc)}}
)
print(f"✅ Inserted {len(chunk_records)} chunk embeddings into '{settings.DOCUMENT_CHUNKS_COLLECTION}' and updated doc status to 'completed'!")

2026-08-15 18:58:48.327 | DEBUG    | app.services.embeddings:split_text:37 - Splitting text into chunks
2026-08-15 18:58:48.327 | INFO     | app.services.embeddings:split_text:42 - Text split completed


Generated 1 chunks for document 'turbine_maintenance_sop.txt'


2026-08-15 18:58:49.468 | DEBUG    | app.services.embeddings:embed_text:46 - Embedding single chunk


✅ Document metadata inserted: ID = 6a80699074408963e3a0f88a


2026-08-15 18:58:53.885 | DEBUG    | app.services.embeddings:embed_text:55 - Chunk embedded successfully


✅ Inserted 1 chunk embeddings into 'document_chunks' and updated doc status to 'completed'!


In [10]:
# 3. Verify querying stored chunks from MongoDB
stored_chunks = await db[settings.DOCUMENT_CHUNKS_COLLECTION].find({"document_id": document_id}).to_list(length=10)
print(f"🔍 Retrieved {len(stored_chunks)} stored chunks for document ID: {document_id}")
for c in stored_chunks:
    print(f" - Chunk #{c['chunk_index']} | ID: {c['chunk_id']} | Vector Dim: {len(c['embedding'])} | Text: {c['text'][:60]}...")

🔍 Retrieved 1 stored chunks for document ID: 6a80699074408963e3a0f88a
 - Chunk #0 | ID: fdf76a23-bfda-4a56-9344-cc4dc59ccd60 | Vector Dim: 1024 | Text: # Gas Turbine Operating Guide
Section 1: Daily inspections o...


## 6. Cleanup Test Data (Optional)

In [11]:
# Run this cell to delete the test data created in Step 5
del_chunks = await db[settings.DOCUMENT_CHUNKS_COLLECTION].delete_many({"document_id": document_id})
del_doc = await db.documents_metadata.delete_one({"_id": document_id})
del_eq = await db.equipment.delete_one({"_id": equipment_id})

print(f"🧹 Cleaned up {del_chunks.deleted_count} chunks, {del_doc.deleted_count} doc metadata, {del_eq.deleted_count} equipment record.")

🧹 Cleaned up 1 chunks, 1 doc metadata, 1 equipment record.
